# L4: Tools for a Customer Outreach Campaign

In this lesson, you will learn more about Tools. You'll focus on three key elements of Tools:
- Versatility
- Fault Tolerance
- Caching

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM
- [Serper](https://serper.dev)

In [2]:
import os
# 直接传 Memory 给 Crew 时，需关闭遥测（否则会报 Invalid type Memory）
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"

from crewai import Agent, Task, Crew, LLM

**Note**: 
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [3]:
from utils import load_env

load_env()



## Creating Agents

In [4]:
sales_rep_agent = Agent(
    role="Sales Representative",
    goal="识别符合我们理想客户画像的高价值潜在客户",
    backstory=(
        "作为 CrewAI 活力销售团队的一员，"
        "你的使命是在数字世界中搜寻潜在客户。"
        "凭借先进工具和战略思维，你分析数据、"
        "趋势与互动，发掘他人可能忽视的机会。"
        "你的工作对促成有意义的接触"
        "并推动公司增长至关重要。"
    ),
    allow_delegation=False,
    verbose=True
)

In [5]:
lead_sales_rep_agent = Agent(
    role="Lead Sales Representative",
    goal="通过个性化、有吸引力的沟通培育潜在客户",
    backstory=(
        "在 CrewAI 销售部门的活跃生态中，"
        "你是潜在客户与其所需解决方案之间的桥梁。"
        "通过撰写引人入胜的个性化信息，"
        "你不仅向潜在客户介绍我们的产品，"
        "还让他们感到被重视与倾听。"
        "你在将兴趣转化为行动方面至关重要，"
        "引导客户从好奇走向承诺。"
    ),
    allow_delegation=False,
    verbose=True
)

## Creating Tools

### crewAI Tools

In [6]:
from crewai_tools import DirectoryReadTool, FileReadTool
from utils import BaiduWebSearchTool

In [7]:
directory_read_tool = DirectoryReadTool(directory='./instructions')
file_read_tool = FileReadTool()
search_tool = BaiduWebSearchTool()

### Custom Tool
- Create a custom tool using crewAi's [BaseTool](https://docs.crewai.com/core-concepts/Tools/#subclassing-basetool) class

In [8]:
from crewai.tools import BaseTool

- Every Tool needs to have a `name` and a `description`.
- For simplicity and classroom purposes, `SentimentAnalysisTool` will return `positive` for every text.
- When running locally, you can customize the code with your logic in the `_run` function.

In [9]:
class SentimentAnalysisTool(BaseTool):
    name: str ="Sentiment Analysis Tool"
    description: str = ("Analyzes the sentiment of text "
         "to ensure positive and engaging communication.")
    
    def _run(self, text: str) -> str:
        # Your custom code tool goes here
        return "positive"

In [10]:
sentiment_analysis_tool = SentimentAnalysisTool()

## Creating Tasks

- The Lead Profiling Task is using crewAI Tools.

In [11]:
lead_profiling_task = Task(
    description=(
        "对 {lead_name} 进行深入分析，"
        "该公司属于 {industry} 行业，"
        "近期对我们的解决方案表现出兴趣。"
        "利用所有可用数据源编制详细画像，"
        "重点关注关键决策者、近期业务动态"
        "以及与我们产品契合的潜在需求。"
        "此任务对有效定制接触策略至关重要。\n"
        "不要做任何假设，"
        "仅使用你完全确定的信息。"
    ),
    expected_output=(
        "关于 {lead_name} 的全面报告，"
        "包括公司背景、关键人员、"
        "近期里程碑及已识别的需求。"
        "突出我们的解决方案可提供价值的领域，"
        "并提出个性化的接触策略。"
    ),
    tools=[directory_read_tool, file_read_tool, search_tool],
    agent=sales_rep_agent,
)

- The Personalized Outreach Task is using your custom Tool `SentimentAnalysisTool`, as well as crewAI's `SerperDevTool` (search_tool).

In [12]:
personalized_outreach_task = Task(
    description=(
        "根据 {lead_name} 潜在客户画像报告中的洞察，"
        "为 {lead_name} 的 {position} {key_decision_maker} "
        "设计个性化外联活动。"
        "活动应回应其近期的 {milestone}，"
        "并说明我们的解决方案如何支持其目标。"
        "你的沟通须与 {lead_name} 的企业文化与价值观相契合，"
        "展现对其业务与需求的深刻理解。\n"
        "不要做任何假设，"
        "仅使用你完全确定的信息。"
    ),
    expected_output=(
        "一系列针对 {lead_name} 的个性化邮件草稿，"
        "专门面向 {key_decision_maker}。"
        "每封草稿应包含引人入胜的叙述，"
        "将我们的解决方案与其近期成就和未来目标联系起来。"
        "确保语气引人入胜、专业，"
        "并与 {lead_name} 的企业形象一致。"
    ),
    tools=[sentiment_analysis_tool, search_tool],
    agent=lead_sales_rep_agent,
)

## Creating the Crew

In [13]:
from crewai.memory.unified_memory import Memory

llm = LLM(model=os.environ.get("OPENAI_MODEL_NAME", "qwen-turbo"))
print("model:", os.environ.get("OPENAI_MODEL_NAME"))

memory_embedder = {
    "provider": "openai",
    "config": {
        "model_name": os.environ.get("EMBEDDINGS_OPENAI_MODEL_NAME", "text-embedding-v3"),
        "api_base": os.environ.get("OPENAI_BASE_URL") or os.environ.get("OPENAI_API_BASE"),
        "api_key": os.environ.get("OPENAI_API_KEY"),
    },
}

crew_memory = Memory(
    llm=llm,
    embedder=memory_embedder,
)

crew = Crew(
    agents=[sales_rep_agent, lead_sales_rep_agent],
    tasks=[lead_profiling_task, personalized_outreach_task],
    verbose=True,
    memory=crew_memory,
)


model: qwen-turbo


## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [14]:
inputs = {
    "lead_name": "DeepLearningAI",
    "industry": "Online Learning Platform",
    "key_decision_maker": "Andrew Ng",
    "position": "CEO",
    "milestone": "product launch"
}

# result = crew.kickoff(inputs=inputs)
result = await crew.kickoff_async(inputs=inputs)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f06ed6c0-536c-4a96-968e-843b5fb9096c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 对 DeepLearningAI 进行深入分析，该公司属于 Online Learning Platform                                      │
│  行业，近期对我们的解决方案表现出兴趣。利用所有可用数据源编制详细画像，重点关注关键决策者、近期业务动态以及与   │
│  我们产品契合的潜在需求。此任务对有效定制接触策略至关重要。                                                     │
│  不要做任何假设，仅使用你完全确定的信息。                                                                       │
│  ID: 38a58ad9-a910-45a1-8351-eab5b4155349                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Sales Representative                                                                                    │
│                                                                                                                 │
│  Task: 对 DeepLearningAI 进行深入分析，该公司属于 Online Learning Platform                                      │
│  行业，近期对我们的解决方案表现出兴趣。利用所有可用数据源编制详细画像，重点关注关键决策者、近期业务动态以及与   │
│  我们产品契合的潜在需求。此任务对有效定制接触策略至关重要。                                                     │
│  不要做任何假设，仅使用你完全确定的信息。                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Sales Representative                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  关于 DeepLearningAI                                                                                            │
│  的全面报告，包括公司背景、关键人员、近期里程碑及已识别的需求。突出我们的解决方案可提供价值的领域，并提出个性   │
│  化的接触策略：                                                                                                 │
│                                                                                                                 │
│  ### 公司背景                                                                                                   │
│  DeepLearningAI 是一家专注于人工智能（AI）教育和培训的在线学习平台。它由知名人工智能专家 Andrew Ng              │
│  创立，致力于为全球的学习者提供高质量的人工智能课程和资源。DeepLearningAI                                       │
│  通过其在线平台，使学习者能够掌握深度学习、机器学习等关键技术，并将其应用于实际项目中。                         │
│                                                                                                                 │
│  ### 关键人员                                                                                                   │
│  - **Andrew Ng**：DeepLearningAI 的创始人兼 CEO。他是一位著名的人工智能专家，曾在 Google Brain                  │
│  和百度担任重要职务，对人工智能领域有深远的影响。                                                               │
│  - **其他关键团队成员**：虽然具体信息未提及，但可以推测 DeepLearningAI                                          │
│  拥有一支由人工智能专家、教育技术专家和内容开发人员组成的团队，以确保其课程的质量和实用性。                     │
│                                                                                                                 │
│  ### 近期里程碑                                                                                                 │
│  - **产品发布**：DeepLearningAI                                                                                 │
│  近期推出了新的课程和产品，进一步巩固了其在在线学习平台领域的领导地位。这些新课程可能涵盖最新的 AI              │
│  技术和应用，如深度学习、自然语言处理等。                                                                       │
│  - **合作伙伴关系**：DeepLearningAI 可能与多家科技公司和研究机构建立了合作关系，以增强其课程内容和技术支持。    │
│                                                                                                                 │
│  ### 已识别的需求                                                                                               │
│  - **高质量的 AI 教育内容**：DeepLearningAI 需要持续提供高质量的 AI 教育内容，以满足全球学习者的需求。          │
│  - **技术支持和工具**：DeepLearningAI 需要强大的技术支持和工具，以确保其在线平台的稳定性和高效性。              │
│  - **用户互动和反馈**：DeepLearningAI 需要有效的用户互动和反馈机制，以不断改进其课程和平台。                    │
│                                                                                                                 │
│  ### 我们的解决方案可提供价值的领域                                                                             │
│  1. **优化内容交付**：我们的解决方案可以帮助 DeepLearningAI 优化其内容交付，提高学习者的体验和满意度。          │
│  2. **提升用户体验**：通过整合 AI 技术和高效的学习分析，我们的解决方案可以提升用户体验，使其更加个性化和高效。  │
│  3. **数据驱动的决策**：我们的平台可以帮助 DeepLearningAI                                                       │
│  实现数据驱动的决策，从而更好地了解用户需求和市场趋势。                                                         │
│                                                                                                                 │
│  ### 个性化的接触策略                                                                                           │
│  1. **个性化邮件**：针对 DeepLearningAI 的 CEO Andrew                                                           │
│  Ng，我们可以设计一封个性化邮件，强调我们的解决方案如何支持其近期的成就和未来目标。邮件内容应引人入胜，展现对   │
│  其业务和需求的深刻理解。                                                                                       │
│  

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Lead Sales Representative                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **个性化邮件草稿一：致敬成就，探讨未来**                                                                       │
│                                                                                                                 │
│  尊敬的 Andrew Ng 博士，                                                                                        │
│                                                                                                                 │
│  我谨代表 CrewAI 团队，向您表达我们对 DeepLearningAI                                                            │
│  最新产品发布的由衷钦佩。作为人工智能教育领域的先驱者，DeepLearningAI 一直以推动全球学习者掌握前沿 AI           │
│  技术为使命，而您的领导力和远见无疑是我们所有人的榜样。                                                         │
│                                                                                                                 │
│  我们注意到，DeepLearningAI 近期推出的课程和产品进一步巩固了其在在线学习平台中的领先地位。这不仅体现了您对 AI   │
│  教育趋势的深刻洞察，也展现了您对学习者体验的持续关注。在这一过程中，DeepLearningAI                             │
│  面临着诸多挑战，例如如何确保高质量内容的高效交付、提升用户体验以及通过数据驱动的决策来优化教学效果。           │
│                                                                                                                 │
│  CrewAI 的解决方案正是为这些需求而生。我们的平台能够帮助 DeepLearningAI                                         │
│  优化内容交付流程，使学习者获得更加流畅和个性化的学习体验。同时，我们提供的 AI                                  │
│  技术支持和数据分析工具，可以助力 DeepLearningAI 实现更精准的用户洞察和更高效的运营决策。                       │
│                                                                                                                 │
│  我们非常期待有机会与您深入交流，探讨如何将 CrewAI 的技术优势与 DeepLearningAI                                  │
│  的愿景相结合，共同推动人工智能教育的未来发展。                                                                 │
│                                                                                                                 │
│  如果您方便的话，我们希望能安排一次简短的会议，进一步了解您的目标，并分享我们能如何提供支持。                   │
│                                                                                                                 │
│  感谢您抽出时间阅读此信。我们真诚希望与 DeepLearningAI 合作，共创卓越。                                         │
│                                                                                                                 │
│  祝好，                                                                                                         │
│  [你的姓名]                                                                                                     │
│  [你的职位]                                                                                                     │
│  CrewAI 团队                                                                                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **个性化邮件草稿二：聚焦创新，携手共赢**                                                                       │
│                                                                                                                 │
│  尊敬的 Andrew Ng 博士，                                                                                        │
│                                                                                    

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 根据 DeepLearningAI 潜在客户画像报告中的洞察，为 DeepLearningAI 的 CEO Andrew Ng                         │
│  设计个性化外联活动。活动应回应其近期的 product launch，并说明我们的解决方案如何支持其目标。你的沟通须与        │
│  DeepLearningAI 的企业文化与价值观相契合，展现对其业务与需求的深刻理解。                                        │
│  不要做任何假设，仅使用你完全确定的信息。                                                                       │
│  Agent: Lead Sales Representative                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: f06ed6c0-536c-4a96-968e-843b5fb9096c                                                                       │
│  Final Output: **个性化邮件草稿一：致敬成就，探讨未来**                                                         │
│                                                                                                                 │
│  尊敬的 Andrew Ng 博士，                                                                                        │
│                                                                                                                 │
│  我谨代表 CrewAI 团队，向您表达我们对 DeepLearningAI                                                            │
│  最新产品发布的由衷钦佩。作为人工智能教育领域的先驱者，DeepLearningAI 一直以推动全球学习者掌握前沿 AI           │
│  技术为使命，而您的领导力和远见无疑是我们所有人的榜样。                                                         │
│                                                                                                                 │
│  我们注意到，DeepLearningAI 近期推出的课程和产品进一步巩固了其在在线学习平台中的领先地位。这不仅体现了您对 AI   │
│  教育趋势的深刻洞察，也展现了您对学习者体验的持续关注。在这一过程中，DeepLearningAI                             │
│  面临着诸多挑战，例如如何确保高质量内容的高效交付、提升用户体验以及通过数据驱动的决策来优化教学效果。           │
│                                                                                                                 │
│  CrewAI 的解决方案正是为这些需求而生。我们的平台能够帮助 DeepLearningAI                                         │
│  优化内容交付流程，使学习者获得更加流畅和个性化的学习体验。同时，我们提供的 AI                                  │
│  技术支持和数据分析工具，可以助力 DeepLearningAI 实现更精准的用户洞察和更高效的运营决策。                       │
│                                                                                                                 │
│  我们非常期待有机会与您深入交流，探讨如何将 CrewAI 的技术优势与 DeepLearningAI                                  │
│  的愿景相结合，共同推动人工智能教育的未来发展。                                                                 │
│                                                                                                                 │
│  如果您方便的话，我们希望能安排一次简短的会议，进一步了解您的目标，并分享我们能如何提供支持。                   │
│                                                                                                                 │
│  感谢您抽出时间阅读此信。我们真诚希望与 DeepLearningAI 合作，共创卓越。                                         │
│                                                                                                                 │
│  祝好，                                                                                                         │
│  [你的姓名]                                                                                                     │
│  [你的职位]                                                                                                     │
│  CrewAI 团队                                                                                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **个性化邮件草稿二：聚焦创新，携手共赢**                                                                       │
│                                                                                                                 │
│  尊敬的 Andrew Ng 博士，                                                                                        │
│                                                                                   

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

- Display the final result as Markdown.

In [15]:
from IPython.display import Markdown
Markdown(result.raw)

**个性化邮件草稿一：致敬成就，探讨未来**

尊敬的 Andrew Ng 博士，

我谨代表 CrewAI 团队，向您表达我们对 DeepLearningAI 最新产品发布的由衷钦佩。作为人工智能教育领域的先驱者，DeepLearningAI 一直以推动全球学习者掌握前沿 AI 技术为使命，而您的领导力和远见无疑是我们所有人的榜样。

我们注意到，DeepLearningAI 近期推出的课程和产品进一步巩固了其在在线学习平台中的领先地位。这不仅体现了您对 AI 教育趋势的深刻洞察，也展现了您对学习者体验的持续关注。在这一过程中，DeepLearningAI 面临着诸多挑战，例如如何确保高质量内容的高效交付、提升用户体验以及通过数据驱动的决策来优化教学效果。

CrewAI 的解决方案正是为这些需求而生。我们的平台能够帮助 DeepLearningAI 优化内容交付流程，使学习者获得更加流畅和个性化的学习体验。同时，我们提供的 AI 技术支持和数据分析工具，可以助力 DeepLearningAI 实现更精准的用户洞察和更高效的运营决策。

我们非常期待有机会与您深入交流，探讨如何将 CrewAI 的技术优势与 DeepLearningAI 的愿景相结合，共同推动人工智能教育的未来发展。

如果您方便的话，我们希望能安排一次简短的会议，进一步了解您的目标，并分享我们能如何提供支持。

感谢您抽出时间阅读此信。我们真诚希望与 DeepLearningAI 合作，共创卓越。

祝好，  
[你的姓名]  
[你的职位]  
CrewAI 团队  

---

**个性化邮件草稿二：聚焦创新，携手共赢**

尊敬的 Andrew Ng 博士，

DeepLearningAI 在人工智能教育领域的持续创新令人印象深刻。从您创立以来，DeepLearningAI 不仅成为全球学习者的首选平台，更在塑造 AI 教育的未来方向上发挥了关键作用。近期的产品发布，尤其是针对最新 AI 技术的课程，彰显了 DeepLearningAI 对行业趋势的敏锐把握。

在这一过程中，DeepLearningAI 需要不断优化其内容交付方式，以满足全球学习者日益增长的需求。同时，提升用户体验和实现数据驱动的决策也是 DeepLearningAI 可持续发展的关键因素。

CrewAI 深知这些挑战，并致力于提供能够真正解决问题的解决方案。我们的平台不仅能够帮助 DeepLearningAI 提升内容交付效率，还能通过先进的 AI 分析技术，为 DeepLearningAI 提供更精准的用户洞察和业务决策支持。

我们诚挚地邀请您参与一次对话，探讨如何将 CrewAI 的能力与 DeepLearningAI 的愿景深度融合，共同打造更加高效、智能的 AI 教育生态。

如您方便，我们期待与您预约一次简短的会议，以便进一步交流。

感谢您的关注。我们相信，通过合作，我们可以为 DeepLearningAI 带来更大的价值。

祝好，  
[你的姓名]  
[你的职位]  
CrewAI 团队  

---

**个性化邮件草稿三：强调价值，展望未来**

尊敬的 Andrew Ng 博士，

DeepLearningAI 的每一次突破都令人振奋，尤其是在人工智能教育领域，您所引领的变革正在改变无数学习者的生活。我们深知，DeepLearningAI 正在通过不断推出高质量的课程和产品，巩固其在全球 AI 教育市场的领导地位。

与此同时，DeepLearningAI 也在面对一些关键挑战，包括如何确保内容的高效交付、提升用户体验，以及利用数据驱动的方式优化教学策略。这些挑战是任何领先企业都会面临的，但它们也提供了巨大的机遇。

CrewAI 的解决方案正是为了应对这些挑战而设计。我们不仅可以帮助 DeepLearningAI 优化内容交付流程，还可以通过 AI 技术提升学习者的个性化体验，同时通过数据分析支持更科学的决策。

我们非常期待与您建立联系，探讨如何将 CrewAI 的能力融入 DeepLearningAI 的战略中，共同推动 AI 教育的发展。

如果方便，我们希望能安排一次简短的会议，深入了解您的目标，并分享我们如何协助您实现更大的影响力。

感谢您抽出时间阅读此信。我们衷心希望与 DeepLearningAI 合作，共创未来。

祝好，  
[你的姓名]  
[你的职位]  
CrewAI 团队